# correctness of the clock implementations

the ground-truth graph is built from program order and message delivery only. both clocks are scored against it.

In [1]:
from pathlib import Path

import pandas as pd

from causality_bench.provenance import read_csv

pd.set_option("display.width", 200)

# results/ sits beside notebooks/ in the repo root
RESULTS = Path.cwd().parent / "results"


def raw(name):
    return read_csv(RESULTS / "raw" / f"{name}.csv")


def processed(name):
    return read_csv(RESULTS / "processed" / f"{name}.csv")


per scenario, how often each clock's verdict differs from the ground truth.

In [2]:
import itertools

from causality_bench.causality import CausalGraph, Relation, lamport_relation, vector_relation
from causality_bench.experiments import all_scenarios

rows = []
for scenario in all_scenarios():
    graph = CausalGraph(scenario.events)
    ordered = concurrent = lamport_wrong = vector_wrong = 0

    for a, b in itertools.combinations(scenario.events, 2):
        truth = graph.relation(a.event_id, b.event_id)
        ordered += truth is not Relation.CONCURRENT
        concurrent += truth is Relation.CONCURRENT
        lamport_wrong += lamport_relation(a, b) is not truth
        vector_wrong += vector_relation(a, b) is not truth

    rows.append({
        "scenario": scenario.name,
        "events": len(scenario.events),
        "ordered_pairs": ordered,
        "concurrent_pairs": concurrent,
        "lamport_disagreements": lamport_wrong,
        "vector_disagreements": vector_wrong,
    })

pd.DataFrame(rows)

,scenario,events,ordered_pairs,concurrent_pairs,lamport_disagreements,vector_disagreements
0,local_causal_chain,3,3,0,0,0
1,message_causality,4,6,0,0,0
2,concurrent_events,2,0,1,0,0
3,multiple_concurrent_nodes,11,13,42,29,0
4,reordered_delivery,4,6,0,0,0
5,causal_chain_across_nodes,7,17,4,3,0


one concrete pair, with the two lamport timestamps, the ground truth, and what the vector comparison says.

In [3]:
from causality_bench.experiments.scenarios import causal_chain_across_nodes

scenario = causal_chain_across_nodes()
graph = CausalGraph(scenario.events)
isolated, receive_b = scenario.event("isolated"), scenario.event("receive_b")

print("L(isolated)  =", isolated.lamport_timestamp)
print("L(receive_b) =", receive_b.lamport_timestamp)
print("ground truth :", graph.relation(isolated.event_id, receive_b.event_id).value)
print("vector says  :", vector_relation(isolated, receive_b).value)

L(isolated)  = 1
L(receive_b) = 3
ground truth : concurrent
vector says  : concurrent
